# Phase 5 — Shopper Universe Flow Visualization

**Purpose:** Visualize the complete shopper journey through sizes — from trial to repeat to lapse — with ASP annotations to identify which flow and price point to fix.

**This is NOT the standard `08_shopper_flow.ipynb` template.** It is a custom funnel + alluvial analysis built specifically for the sizing/pricing strategy question.

| Step | Description |
|------|-------------|
| 5-1 | Size-level funnel: Category Trial → Sub-brand Trial → Repeat → Lapse (per size, absolute numbers) |
| 5-2 | Alluvial/Sankey: trial size → repeat size → lapse destination (sub-brand + size) |
| 5-3 | ASP-annotated flow: overlay prevailing ASP at each node to connect price to flow volume |

**Trial Definition:** Shopper with no purchase of the sub-brand in the prior **12 months (365 days)** — uses `LAG()` window function over `LOOKBACK_START` to `ANALYSIS_END`.  
**Repeat/Lapse Window:** 180 days (6 months) after trial date.

**Created:** 2026-02-19

---
## 0. Imports & Connection

In [1]:
import os
import pandas as pd
import numpy as np
import warnings
from dotenv import load_dotenv
import databricks.sql as sql
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.1f}')

def _find_japanese_font():
    for name in ['MS Gothic', 'MS PGothic', 'Yu Gothic', 'Meiryo', 'IPAexGothic']:
        if name in {f.name for f in fm.fontManager.ttflist}:
            return name
    return None
_jp_font = _find_japanese_font()
if _jp_font:
    plt.rcParams['font.family'] = _jp_font
    print(f'✅ Japanese font: {_jp_font}')

load_dotenv(dotenv_path='../../.env')
DATABRICKS_HOST      = os.getenv('DATABRICKS_HOST')
DATABRICKS_TOKEN     = os.getenv('DATABRICKS_TOKEN')
DATABRICKS_HTTP_PATH = os.getenv('DATABRICKS_HTTP_PATH')
assert all([DATABRICKS_HOST, DATABRICKS_TOKEN, DATABRICKS_HTTP_PATH]), 'Missing .env credentials'
print('✅ Credentials loaded')

def execute_query(query: str) -> pd.DataFrame:
    with sql.connect(server_hostname=DATABRICKS_HOST, http_path=DATABRICKS_HTTP_PATH,
                     access_token=DATABRICKS_TOKEN) as conn:
        with conn.cursor() as cur:
            cur.execute(query)
            result = cur.fetchall()
            columns = [d[0] for d in cur.description]
            return pd.DataFrame(result, columns=columns)

✅ Japanese font: MS Gothic
✅ Credentials loaded


---
## 1. Parameters

In [2]:
ARIEL_GEL   = 'ｱﾘｴｰﾙｼﾞｪﾙ'
ATTACK_EX   = 'ｱﾀｯｸ抗菌EX'
SUB_CAT     = '洗濯洗剤'
CATEGORY    = 'Laundry'

LOOKBACK_START    = '2024-01-01'
ANALYSIS_START    = '2025-01-01'
ANALYSIS_END      = '2026-01-31'
RENEWAL_MONTH     = '2025-05-01'

# ── Canonical parameters (shared across NB00–NB05) ───────────────────
TRIAL_LOOKBACK_DAYS      = 365   # 12-month lookback for trial definition
REPEAT_LAPSE_WINDOW_DAYS = 180   # 6-month repeat / lapse window
LAPSE_WINDOW_DAYS        = REPEAT_LAPSE_WINDOW_DAYS  # backward compat alias

LATEST_COHORT_END = '2025-07-31'

RETAILER_CODES = [
    'cds_8005', 'cds_8006', 'cds_8007', 'cds_8008', 'cds_8009',
    'cds_8010', 'cds_8011', 'cds_8012', 'cds_8013',
]
RETAILER_IN = ', '.join(f"'{c}'" for c in RETAILER_CODES)

# ── Size order (physical size: small → large) and exclusions ──────────
SIZE_ORDER     = ['本体通常', '詰替超特大', '詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ', '詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ', '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ']
EXCLUDED_SIZES = ['ｿﾉﾀ', '詰替通常', '詰替超ｼﾞｬﾝﾎﾞ']

def order_and_filter_sizes(sizes_list):
    """Return sizes in SIZE_ORDER, excluding EXCLUDED_SIZES."""
    sizes_set = set(sizes_list) - set(EXCLUDED_SIZES)
    return [s for s in SIZE_ORDER if s in sizes_set]

print(f'📋 Flow visualization: {ANALYSIS_START} → {ANALYSIS_END}')
print(f'📋 Size order: {SIZE_ORDER}')


📋 Flow visualization: 2025-01-01 → 2026-01-31
📋 Size order: ['本体通常', '詰替超特大', '詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ', '詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ', '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ']


### Canonical Definitions (shared across NB00–NB05)

| Concept | Rule | Parameter |
|---------|------|-----------|
| **Trial shopper** | No purchase of the sub-brand in the prior **12 months (365 days)** — detected via `LAG()` window function | `TRIAL_LOOKBACK_DAYS = 365` |
| **Repeat shopper** | ≥1 additional purchase within **180 days** of trial | `REPEAT_LAPSE_WINDOW_DAYS = 180` |
| **Lapsed shopper** | Zero purchases within **180 days** of last purchase | `REPEAT_LAPSE_WINDOW_DAYS = 180` |
| **ASP bin** | `FLOOR(ASP / 50) * 50` — 50 JPY floor bins | x-axis: "ASP (50 JPY bin)" |

---
## 2. Build Complete Shopper Journey Dataset

One unified query that tracks each shopper's journey:
- Category trial OR existing category buyer
- Sub-brand trial (first Ariel Gel purchase)
- Entry size
- Repeat event (if any) and repeat size
- Lapse event and destination

In [4]:
# ── Extended-timeout helper for long-running queries ──────────────────
def execute_query_long(query: str) -> pd.DataFrame:
    """Like execute_query but with 1-hour retry timeout."""
    with sql.connect(
        server_hostname=DATABRICKS_HOST, http_path=DATABRICKS_HTTP_PATH,
        access_token=DATABRICKS_TOKEN,
        _retry_stop_after_attempts_duration=3600
    ) as conn:
        with conn.cursor() as cur:
            cur.execute(query)
            result = cur.fetchall()
            columns = [d[0] for d in cur.description]
            return pd.DataFrame(result, columns=columns)

# ── Journey query — LAG()-based 12-month lookback trial definition ────
# HARMONIZED with NB02/NB03: trial = no purchase of sub-brand in prior
# 365 days, detected via LAG() window function.
# Scans from LOOKBACK_START (2024-01-01) to build purchase history,
# then filters trial events to ANALYSIS_START..LATEST_COHORT_END.
journey_query = f"""
WITH all_ariel AS (
    SELECT
        idpos.shopper_key,
        prod.jp_segment_4_name AS size_code,
        CAST(idpos.sales_period_group_end_date_part AS DATE) AS purchase_date,
        SUM(idpos.pos_sales_amt) / SUM(idpos.pos_unit_sales_qty) AS asp
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    INNER JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
    INNER JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
    WHERE idpos.sales_period_group_end_date_part BETWEEN '{LOOKBACK_START}' AND '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}'
      AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND prod.jp_sub_brand_alter_lang_name = '{ARIEL_GEL}'
      AND shopper.member_ind = 'Y'
      AND idpos.pos_unit_sales_qty > 0
    GROUP BY 1, 2, 3
),
-- LAG()-based trial detection: gap > 365 days OR first-ever purchase
with_prev AS (
    SELECT *,
           LAG(purchase_date) OVER (
               PARTITION BY shopper_key ORDER BY purchase_date
           ) AS prev_purchase_date
    FROM all_ariel
),
trial_candidates AS (
    SELECT *
    FROM with_prev
    WHERE purchase_date BETWEEN '{ANALYSIS_START}' AND '{LATEST_COHORT_END}'
      AND (prev_purchase_date IS NULL
           OR DATEDIFF(purchase_date, prev_purchase_date) > {TRIAL_LOOKBACK_DAYS})
),
-- Keep earliest qualifying trial event per shopper
trial_events AS (
    SELECT shopper_key,
           MIN(purchase_date) AS trial_date
    FROM trial_candidates
    GROUP BY 1
),
-- Trial shoppers with entry size + ASP
trial_shoppers AS (
    SELECT
        te.shopper_key,
        te.trial_date,
        aa.size_code AS trial_size,
        aa.asp       AS trial_asp
    FROM trial_events te
    INNER JOIN all_ariel aa
           ON te.shopper_key = aa.shopper_key
          AND te.trial_date  = aa.purchase_date
),
-- Find first repeat within REPEAT_LAPSE_WINDOW_DAYS
with_repeat AS (
    SELECT
        ts.*,
        aa.purchase_date AS repeat_date,
        aa.size_code     AS repeat_size,
        aa.asp           AS repeat_asp,
        ROW_NUMBER() OVER (PARTITION BY ts.shopper_key ORDER BY aa.purchase_date) AS rn
    FROM trial_shoppers ts
    LEFT JOIN all_ariel aa
           ON ts.shopper_key = aa.shopper_key
          AND aa.purchase_date > ts.trial_date
          AND aa.purchase_date <= DATE_ADD(ts.trial_date, {REPEAT_LAPSE_WINDOW_DAYS})
)
SELECT
    shopper_key,
    'Trial' AS cat_status,
    trial_date,
    trial_size,
    trial_asp,
    repeat_date,
    repeat_size,
    repeat_asp,
    CASE WHEN repeat_date IS NOT NULL THEN 'Repeat' ELSE 'Lapse' END AS outcome
FROM with_repeat
WHERE rn = 1 OR repeat_date IS NULL
"""

print('⏳ Building complete shopper journey dataset (LAG-based trial, harmonized)...', flush=True)
df_journey = execute_query_long(journey_query)
df_journey['trial_date'] = pd.to_datetime(df_journey['trial_date'])
df_journey['repeat_date'] = pd.to_datetime(df_journey['repeat_date'])
for col in ['trial_asp', 'repeat_asp']:
    df_journey[col] = pd.to_numeric(df_journey[col])

print(f'✅ {len(df_journey):,} shopper journeys loaded')
print(f'   Repeat: {(df_journey["outcome"]=="Repeat").sum():,}')
print(f'   Lapse:  {(df_journey["outcome"]=="Lapse").sum():,}')

⏳ Building complete shopper journey dataset (LAG-based trial, harmonized)...


HTTP request error: 'NoneType' object has no attribute 'request'


✅ 1,651,373 shopper journeys loaded
   Repeat: 558,295
   Lapse:  1,093,078


In [5]:
# ── Load lapse destinations from Phase 4 export (avoid re-querying) ───
# Phase 4 already computed the full lapse destination analysis
p4_flow = pd.read_excel('phase4_lapse_analysis.xlsx', sheet_name='Flow_Detail')
p4_dest = pd.read_excel('phase4_lapse_analysis.xlsx', sheet_name='Destination_Summary')

print(f'✅ Loaded Phase 4 lapse destination data')
print(f'   Flow detail rows: {len(p4_flow):,}')
print(f'   Destination summary rows: {len(p4_dest):,}')
print(f'   Top 5 destinations:')
print(p4_dest.head().to_string(index=False))

✅ Loaded Phase 4 lapse destination data
   Flow detail rows: 893
   Destination summary rows: 168
   Top 5 destinations:
next_sub_brand     next_size  shoppers  share_%
      ｱﾀｯｸ抗菌EX 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ    141498      7.9
      ｱﾀｯｸ抗菌EX         詰替超特大     87689      4.9
      ｱﾀｯｸ抗菌EX  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     81855      4.6
      ｱﾀｯｸ抗菌EX   詰替ﾒｶﾞｼﾞｬﾝﾎﾞ     58705      3.3
      ｱﾀｯｸ抗菌EX          本体通常     43455      2.4


---
## 3. Step 5-1: Size-Level Funnel — Category Trial → Sub-brand Trial → Repeat → Lapse

In [6]:
# ── Build funnel data per trial entry size ────────────────────────────
# Filter excluded sizes from journey data
df_journey_filtered = df_journey[~df_journey['trial_size'].isin(EXCLUDED_SIZES)].copy()

funnel_data = df_journey_filtered.groupby('trial_size').agg(
    total_trial     = ('shopper_key', 'nunique'),
    repeat_shoppers = ('outcome', lambda x: (x == 'Repeat').sum()),
    lapse_shoppers  = ('outcome', lambda x: (x == 'Lapse').sum()),
    avg_trial_asp   = ('trial_asp', 'mean'),
).reset_index()

funnel_data['repeat_rate_%'] = (funnel_data['repeat_shoppers'] / funnel_data['total_trial'] * 100).round(1)
funnel_data['lapse_rate_%']  = (funnel_data['lapse_shoppers'] / funnel_data['total_trial'] * 100).round(1)

# Reorder by SIZE_ORDER
funnel_data['_sort'] = funnel_data['trial_size'].map(
    {s: i for i, s in enumerate(SIZE_ORDER)}
).fillna(99)
funnel_data = funnel_data.sort_values('_sort').drop(columns='_sort')

print('Funnel Summary per Entry Size:')
print('=' * 90)
print(funnel_data.to_string(index=False))


Funnel Summary per Entry Size:
   trial_size  total_trial  repeat_shoppers  lapse_shoppers  avg_trial_asp  repeat_rate_%  lapse_rate_%
         本体通常       409376           141322          268054          261.3           34.5          65.5
        詰替超特大       471574           173731          297843          331.2           36.8          63.2
 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ       289540            95145          194395          642.0           32.9          67.1
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ       328785           108066          220719          863.7           32.9          67.1
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ       137018            36408          100610          967.3           26.6          73.4


In [7]:
# ── Bar chart: trial universe and outcome per size ────────────────────
# Sizes ordered by SIZE_ORDER (already sorted in funnel_data)
sizes = order_and_filter_sizes(funnel_data['trial_size'].unique())
# Reverse for horizontal bar (smallest at top so largest lands bottom-right)
sizes_display = list(reversed(sizes))

fd = funnel_data.set_index('trial_size')

# ── Trial universe bar ────────────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Bar(
    y=sizes_display,
    x=fd.loc[sizes_display, 'total_trial'],
    name='Trial Shoppers',
    orientation='h', marker_color='#6495ED',
    text=fd.loc[sizes_display, 'total_trial'],
    textposition='inside'
))
fig.update_layout(
    title='Trial Shopper Universe by Entry Size',
    xaxis_title='Number of Shoppers',
    template='plotly_white', height=400,
    legend=dict(orientation='h', yanchor='bottom', y=-0.3)
)
fig.show()

# ── Repeat vs Lapse per size ──────────────────────────────────────────
fig2 = go.Figure()

fig2.add_trace(go.Bar(
    y=sizes_display,
    x=fd.loc[sizes_display, 'repeat_shoppers'],
    name='Repeat',
    orientation='h', marker_color='#2E8B57',
    text=[f"{v:,} ({r:.1f}%)" for v, r in zip(
        fd.loc[sizes_display, 'repeat_shoppers'],
        fd.loc[sizes_display, 'repeat_rate_%'])],
    textposition='inside'
))
fig2.add_trace(go.Bar(
    y=sizes_display,
    x=fd.loc[sizes_display, 'lapse_shoppers'],
    name='Lapse (6-month no return)',
    orientation='h', marker_color='#CC3333',
    text=[f"{v:,} ({r:.1f}%)" for v, r in zip(
        fd.loc[sizes_display, 'lapse_shoppers'],
        fd.loc[sizes_display, 'lapse_rate_%'])],
    textposition='inside'
))

fig2.update_layout(
    barmode='stack',
    title='Trial Outcome by Entry Size — Where Do We Lose Shoppers?',
    xaxis_title='Number of Shoppers',
    template='plotly_white', height=400,
    legend=dict(orientation='h', yanchor='bottom', y=-0.3)
)

# Add ASP annotation
for i, size in enumerate(sizes_display):
    row = funnel_data[funnel_data['trial_size'] == size].iloc[0]
    fig2.add_annotation(x=row['total_trial'] + 50, y=size,
                        text=f"¥{row['avg_trial_asp']:,.0f}",
                        showarrow=False, font=dict(size=11, color='#333'))

fig2.show()


---
## 4. Step 5-2: Alluvial/Sankey — Trial Size → Repeat Size → Lapse Destination

In [8]:
# ── Prepare flow data from journey + Phase 4 lapse destinations ───────
# For repeat shoppers: destination = Ariel + repeat_size
# For lapse shoppers: use Phase 4 aggregated flow data

# Stage 1 flow: trial_size → outcome
flow_stage1 = df_journey.groupby(['trial_size', 'outcome']).agg(
    count=('shopper_key', 'nunique'),
    avg_asp=('trial_asp', 'mean')
).reset_index()

# Stage 2 flow (Repeat branch): outcome → Ariel repeat_size
repeat_flow = df_journey[df_journey['outcome'] == 'Repeat'].groupby('repeat_size').agg(
    count=('shopper_key', 'nunique')
).reset_index()
repeat_flow.columns = ['dest_label', 'count']
repeat_flow['dest_label'] = 'Ariel ' + repeat_flow['dest_label']
repeat_flow['outcome'] = 'Repeat'

# Stage 2 flow (Lapse branch): outcome → destination (from Phase 4)
lapse_dest = p4_dest.copy()
lapse_dest['dest_label'] = lapse_dest['next_sub_brand'] + ' ' + lapse_dest['next_size'].fillna('')
lapse_dest = lapse_dest.rename(columns={'shoppers': 'count'})
# Add category exit
total_lapsed_journey = (df_journey['outcome'] == 'Lapse').sum()
total_tracked_dest = lapse_dest['count'].sum()
category_exit = max(0, total_lapsed_journey - total_tracked_dest)
lapse_dest = pd.concat([
    lapse_dest[['dest_label', 'count']],
    pd.DataFrame([{'dest_label': 'Category Exit', 'count': category_exit}])
], ignore_index=True)
lapse_dest['outcome'] = 'Lapse'

# Simplify to top destinations
top_dests_all = pd.concat([repeat_flow, lapse_dest]).nlargest(15, 'count')['dest_label'].tolist()

flow_stage2 = pd.concat([repeat_flow, lapse_dest], ignore_index=True)
flow_stage2['dest_simplified'] = flow_stage2['dest_label'].apply(
    lambda x: x if x in top_dests_all else 'Other Brands'
)
flow_stage2 = flow_stage2.groupby(['outcome', 'dest_simplified']).agg(count=('count', 'sum')).reset_index()

print('Flow Stage 1 (Trial Size → Outcome):')
print(flow_stage1.to_string(index=False))
print(f'\nFlow Stage 2 (Outcome → Destination) — top entries:')
print(flow_stage2.nlargest(15, 'count').to_string(index=False))

Flow Stage 1 (Trial Size → Outcome):
   trial_size outcome  count  avg_asp
         本体通常   Lapse 268054    257.6
         本体通常  Repeat 141322    268.3
        詰替超特大   Lapse 297843    330.9
        詰替超特大  Repeat 173731    331.6
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ   Lapse 220719    866.5
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ  Repeat 108066    857.9
    詰替超ｼﾞｬﾝﾎﾞ   Lapse   2805    664.3
    詰替超ｼﾞｬﾝﾎﾞ  Repeat    954    684.8
         詰替通常   Lapse     98    203.6
         詰替通常  Repeat     45    179.1
 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ   Lapse 194395    631.3
 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ  Repeat  95145    663.9
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ   Lapse 100610    964.6
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ  Repeat  36408    974.8
          ｿﾉﾀ   Lapse   8554  1,935.5
          ｿﾉﾀ  Repeat   2624  1,928.5

Flow Stage 2 (Outcome → Destination) — top entries:
outcome        dest_simplified  count
  Lapse           Other Brands 390372
  Lapse          Category Exit 200205
 Repeat            Ariel 詰替超特大 177826
  Lapse ｱﾀｯｸ抗菌EX 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ 141498
 Repeat    Ariel 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ 138754
 Repeat             Ariel 本体通常 10527

In [9]:
# ── Build Sankey flow data ────────────────────────────────────────────
# Stage 1: Trial Size → Outcome (Repeat / Lapse)
# Stage 2: Outcome → Destination (repeat size / lapse destination)

trial_sizes = sorted(df_journey['trial_size'].unique())
outcomes = ['Repeat', 'Lapse']
destinations = sorted(flow_stage2['dest_simplified'].unique())

all_nodes = (
    [f'Trial: {s}' for s in trial_sizes] +
    outcomes +
    [f'→ {d}' for d in destinations]
)
node_idx = {name: i for i, name in enumerate(all_nodes)}

# Build links
sources, targets, values, colors = [], [], [], []

# Stage 1: Trial Size → Outcome
for _, row in flow_stage1.iterrows():
    src = node_idx[f'Trial: {row["trial_size"]}']
    tgt = node_idx[row['outcome']]
    sources.append(src)
    targets.append(tgt)
    values.append(row['count'])
    colors.append('rgba(46,139,87,0.4)' if row['outcome'] == 'Repeat' else 'rgba(204,51,51,0.4)')

# Stage 2: Outcome → Destination
for _, row in flow_stage2.iterrows():
    src = node_idx[row['outcome']]
    tgt = node_idx[f'→ {row["dest_simplified"]}']
    sources.append(src)
    targets.append(tgt)
    values.append(row['count'])
    if ARIEL_GEL in row['dest_simplified'] or 'Ariel' in row['dest_simplified']:
        colors.append('rgba(30,144,255,0.4)')  # Ariel = blue
    elif ATTACK_EX in row['dest_simplified']:
        colors.append('rgba(255,99,71,0.5)')   # Attack = red
    elif 'Exit' in row['dest_simplified']:
        colors.append('rgba(128,128,128,0.3)') # Exit = gray
    else:
        colors.append('rgba(255,165,0,0.3)')   # Other = orange

# Node colors
node_colors = (
    ['#1E90FF'] * len(trial_sizes) +  # Trial sizes = blue
    ['#2E8B57', '#CC3333'] +           # Repeat = green, Lapse = red
    ['#6495ED'] * len(destinations)    # Destinations = light blue
)

fig = go.Figure(go.Sankey(
    node=dict(
        pad=15, thickness=20,
        label=all_nodes,
        color=node_colors
    ),
    link=dict(
        source=sources, target=targets, value=values,
        color=colors
    )
))

fig.update_layout(
    title_text='Shopper Universe Flow: Trial Size → Repeat/Lapse → Destination (Sub-brand + Size)',
    font_size=11, height=700, template='plotly_white'
)
fig.show()

In [10]:
# ── (NEW) アタック抗菌EX Sankey: Trial Size → Repeat/Lapse → Destination ─
# Load Attack cohort from phase 3 export and lapse destinations from phase 4
import os

# Load data
try:
    df_attack_cohort = pd.read_excel('phase3_trial_repeat.xlsx', sheet_name='Cohort_Funnel')
    df_attack_cohort = df_attack_cohort[df_attack_cohort['sub_brand'] == ATTACK_EX].copy()
    # Also try to load Attack lapse flow from phase 4
    if os.path.exists('phase4_lapse_analysis.xlsx'):
        xl4 = pd.ExcelFile('phase4_lapse_analysis.xlsx')
        sheets = xl4.sheet_names
        print(f'Phase 4 sheets available: {sheets}')
        # Load flow detail if available
        if 'Flow_Detail' in sheets:
            flow_all = pd.read_excel('phase4_lapse_analysis.xlsx', sheet_name='Flow_Detail')
            # Check if Attack data is in here (it would be from Ariel lapse only)
            p4_attack_dest = None  # Attack dest not in original flow_detail
        else:
            p4_attack_dest = None
    else:
        p4_attack_dest = None
    print(f'✅ Attack cohort loaded: {len(df_attack_cohort)} rows')
except Exception as e:
    print(f'⚠️ Could not load from Excel: {e}')
    print('💡 Run phase 3 notebook first to generate phase3_trial_repeat.xlsx')
    df_attack_cohort = None

if df_attack_cohort is not None and len(df_attack_cohort) > 0:
    # Reconstruct journey-style from cohort funnel pivot
    # Try loading full cohort data
    try:
        df_atk_full = pd.read_excel('phase3_trial_repeat.xlsx')
        # If there's a raw cohort sheet
        df_atk_journey = df_atk_full[df_atk_full['sub_brand'] == ATTACK_EX].copy() \
            if 'sub_brand' in df_atk_full.columns else None
    except:
        df_atk_journey = None

    # Fall back to cohort funnel summary
    if df_atk_journey is not None and 'trial_size' in df_atk_journey.columns:
        atk_journey = df_atk_journey[~df_atk_journey['trial_size'].isin(EXCLUDED_SIZES)].copy()
    else:
        # Reconstruct from funnel pivot: each outcome row = Repeat or Lapse
        if 'Repeat' in df_attack_cohort.columns and 'Lapse' in df_attack_cohort.columns:
            rows = []
            for _, r in df_attack_cohort.iterrows():
                if r.get('trial_size', None) in EXCLUDED_SIZES:
                    continue
                rows.append({'trial_size': r['trial_size'], 'outcome': 'Repeat',
                              'count': r.get('Repeat', 0)})
                rows.append({'trial_size': r['trial_size'], 'outcome': 'Lapse',
                              'count': r.get('Lapse', 0)})
            atk_flow_stage1 = pd.DataFrame(rows)
        else:
            atk_flow_stage1 = None

    # Build Sankey from atk_flow_stage1 if available
    if 'atk_flow_stage1' in dir() and atk_flow_stage1 is not None and len(atk_flow_stage1) > 0:
        atk_trial_sizes = order_and_filter_sizes(atk_flow_stage1['trial_size'].unique())
        atk_outcomes = ['Repeat', 'Lapse']
        atk_dests = ['→ Repeat Ariel', '→ Other Brand', '→ Category Exit']

        atk_nodes = (
            [f'Attack Trial: {s}' for s in atk_trial_sizes] +
            ['Attack Repeat', 'Attack Lapse']
        )
        atk_node_idx = {name: i for i, name in enumerate(atk_nodes)}

        atk_sources, atk_targets, atk_values, atk_colors = [], [], [], []

        for _, row in atk_flow_stage1.iterrows():
            if row['trial_size'] not in atk_trial_sizes:
                continue
            src_name = f'Attack Trial: {row["trial_size"]}'
            tgt_name = f'Attack {row["outcome"]}'
            if src_name not in atk_node_idx:
                continue
            if tgt_name not in atk_node_idx:
                atk_nodes.append(tgt_name)
                atk_node_idx[tgt_name] = len(atk_nodes) - 1
            atk_sources.append(atk_node_idx[src_name])
            atk_targets.append(atk_node_idx[tgt_name])
            atk_values.append(int(row['count']))
            atk_colors.append('rgba(46,139,87,0.4)' if row['outcome'] == 'Repeat' else 'rgba(204,51,51,0.4)')

        atk_node_colors = (
            ['#FF6347'] * len(atk_trial_sizes) +
            ['#2E8B57', '#CC3333']
        )

        fig_atk_sankey = go.Figure(go.Sankey(
            node=dict(
                pad=15, thickness=20,
                label=atk_nodes,
                color=atk_node_colors + ['#888888'] * (len(atk_nodes) - len(atk_node_colors))
            ),
            link=dict(
                source=atk_sources, target=atk_targets, value=atk_values,
                color=atk_colors
            )
        ))
        fig_atk_sankey.update_layout(
            title_text='アタック抗菌EX Shopper Flow: Trial Size → Repeat / Lapse',
            font_size=11, height=600, template='plotly_white'
        )
        fig_atk_sankey.show()
    else:
        print('⚠️ Could not build Attack Sankey: cohort funnel data not in expected format')
        print('   Re-run notebook 03 to refresh phase3_trial_repeat.xlsx with Attack cohort data')
else:
    print('⚠️ Attack cohort data not available — run notebook 03 first')


Phase 4 sheets available: ['Lapse_Rate_by_Size', 'Lapse_by_ASP_Band', 'Lapse_ASP_AllSizes', 'Destination_Summary', 'Flow_Detail', 'Ariel_to_Attack']
✅ Attack cohort loaded: 7 rows
⚠️ Could not build Attack Sankey: cohort funnel data not in expected format
   Re-run notebook 03 to refresh phase3_trial_repeat.xlsx with Attack cohort data


---
## 5. Step 5-3: ASP-Annotated Flow Summary

Overlay the prevailing ASP at each funnel stage to identify which price point to fix.

In [11]:
# ── ASP at each stage per size ────────────────────────────────────────
asp_by_stage = []

for size in trial_sizes:
    size_data = df_journey[df_journey['trial_size'] == size]
    total        = len(size_data)
    repeat_data  = size_data[size_data['outcome'] == 'Repeat']
    lapse_data   = size_data[size_data['outcome'] == 'Lapse']

    asp_by_stage.append({
        'size': size,
        'trial_shoppers': total,
        'avg_trial_asp': size_data['trial_asp'].mean(),
        'repeat_shoppers': len(repeat_data),
        'repeat_rate_%': round(len(repeat_data) / max(total, 1) * 100, 1),
        'avg_repeat_asp': repeat_data['repeat_asp'].mean() if len(repeat_data) > 0 else None,
        'lapse_shoppers': len(lapse_data),
        'lapse_rate_%': round(len(lapse_data) / max(total, 1) * 100, 1),
        'avg_lapse_trial_asp': lapse_data['trial_asp'].mean() if len(lapse_data) > 0 else None,
    })

df_asp_flow = pd.DataFrame(asp_by_stage)

print('=' * 100)
print('ASP-Annotated Funnel: Which SIZE + PRICE POINT needs intervention?')
print('=' * 100)
print(df_asp_flow.to_string(index=False))

print('\n📊 Interpretation Guide:')
print('  - High lapse_rate_% + high avg_trial_asp = price too high for trial conversion')
print('  - Large lapse_shoppers + low avg_trial_asp = price is not the issue, look at product/competitor')
print('  - avg_repeat_asp < avg_trial_asp = shoppers expect discount to repeat → margin risk')

ASP-Annotated Funnel: Which SIZE + PRICE POINT needs intervention?
         size  trial_shoppers  avg_trial_asp  repeat_shoppers  repeat_rate_%  avg_repeat_asp  lapse_shoppers  lapse_rate_%  avg_lapse_trial_asp
         本体通常          409376          261.3           141322           34.5           419.1          268054          65.5                257.6
        詰替超特大          471574          331.2           173731           36.8           421.3          297843          63.2                330.9
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ          328785          863.7           108066           32.9           784.7          220719          67.1                866.5
    詰替超ｼﾞｬﾝﾎﾞ            3759          669.5              954           25.4           722.6            2805          74.6                664.3
         詰替通常             143          195.9               45           31.5           308.8              98          68.5                203.6
 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ          289540          642.0            95145        

In [12]:
# ── Bubble chart: size universe with ASP and rates ────────────────────
fig = px.scatter(
    df_asp_flow,
    x='avg_trial_asp',
    y='repeat_rate_%',
    size='trial_shoppers',
    text='size',
    color='lapse_rate_%',
    color_continuous_scale='RdYlGn_r',  # Red = high lapse, Green = low lapse
    labels={
        'avg_trial_asp': 'Average Trial ASP (JPY)',
        'repeat_rate_%': 'Repeat Rate (%)',
        'trial_shoppers': 'Trial Shoppers (bubble size)',
        'lapse_rate_%': 'Lapse Rate (%)'
    },
    title='Strategic Map: Which Size + Price Needs Fixing?<br>(Big bubble = big opportunity, Red = high lapse risk)'
)
fig.update_traces(textposition='top center', marker=dict(sizemin=10))
fig.update_layout(template='plotly_white', height=600)
fig.show()

In [13]:
# ── Export Phase 5 results ────────────────────────────────────────────
output_file = 'phase5_shopper_flow.xlsx'

# Strip timezone info to avoid Excel tz-aware error
for col in funnel_data.select_dtypes(include=['datetimetz']).columns:
    funnel_data[col] = funnel_data[col].dt.tz_localize(None)

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    funnel_data.to_excel(writer, sheet_name='Funnel_by_Size', index=False)
    flow_stage1.to_excel(writer, sheet_name='Flow_TrialSize_to_Outcome', index=False)
    flow_stage2.to_excel(writer, sheet_name='Flow_Outcome_to_Dest', index=False)
    df_asp_flow.to_excel(writer, sheet_name='ASP_Annotated_Funnel', index=False)

print(f'✅ Phase 5 results exported to {output_file}')
print('\n🎯 Next step: Strategy Memo — synthesize findings from all 6 phases.')

✅ Phase 5 results exported to phase5_shopper_flow.xlsx

🎯 Next step: Strategy Memo — synthesize findings from all 6 phases.
